# Smoke-эксперимент: японская полярность и эмоции

## tl;dr

На 10 строках synthetic test фактический прогон ниже даёт polarity macro-F1 = 1.000, emotion micro-F1 = 1.000, emotion macro-F1 = 0.750 и Hamming loss = 0.000. Это только техническая проверка; WRIME не загружается и не перераспространяется.

## Context & Methods

Notebook импортирует код из `src/`, читает committed synthetic CSV и вызывает реальные `train()` и `evaluate()`. Общий char TF-IDF используется для классификации полярности и восьми независимых emotion-меток; пороги эмоций выбираются только на validation.

### Key Assumptions

- seed равен 42, split стратифицирован по полярности;
- все emotion-метки бинарные, а в train присутствуют оба значения каждой метки;
- synthetic-фразы не являются строками WRIME;
- внешние данные и сеть не используются.

In [1]:
import sys
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd

SEED = 42
PROJECT_ROOT = next(
    path for path in (Path.cwd(), Path.cwd().parent)
    if (path / 'pyproject.toml').exists()
)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from japanese_emotion_sentiment.data import (  # noqa: E402
    EMOTION_COLUMNS,
    load_csv,
    stratified_split,
)
from japanese_emotion_sentiment.evaluate import evaluate  # noqa: E402
from japanese_emotion_sentiment.predict import predict  # noqa: E402
from japanese_emotion_sentiment.train import train as train_model  # noqa: E402

DATA_PATH = PROJECT_ROOT / 'data' / 'smoke.csv'

## Data

Проверяем размер частей, распределение полярности и число положительных emotion-меток.

In [2]:
texts = load_csv(DATA_PATH)
train_frame, validation_frame, test_frame, split_manifest = stratified_split(
    texts, random_state=SEED
)
assert set(train_frame['text_id']).isdisjoint(validation_frame['text_id'])
assert set(train_frame['text_id']).isdisjoint(test_frame['text_id'])
assert set(validation_frame['text_id']).isdisjoint(test_frame['text_id'])

split_counts = pd.concat({
    'train': train_frame['polarity'].value_counts().sort_index(),
    'validation': validation_frame['polarity'].value_counts().sort_index(),
    'test': test_frame['polarity'].value_counts().sort_index(),
}, axis=1).fillna(0).astype(int)
split_counts

,train,validation,test
polarity,,,
negative,9,4,3
neutral,10,3,3
positive,9,3,4


In [3]:
pd.Series(
    train_frame.loc[:, EMOTION_COLUMNS].sum().to_dict(),
    name='positive labels in train',
).sort_values(ascending=False)

emotion_joy             9
emotion_surprise        8
emotion_sadness         6
emotion_fear            5
emotion_trust           5
emotion_anticipation    3
emotion_anger           1
emotion_disgust         1
Name: positive labels in train, dtype: int64

## Results

Обучаем обе реальные модели, сохраняем bundle и оцениваем сохранённый test artifact.

In [4]:
temporary_directory = TemporaryDirectory(prefix='japanese-emotion-smoke-')
artifact_dir = Path(temporary_directory.name) / 'artifacts'
training_metadata = train_model(DATA_PATH, artifact_dir, random_state=SEED)
test_metrics = evaluate(artifact_dir / 'model.joblib', artifact_dir / 'test.csv')

pd.Series({
    'test_rows': test_metrics['rows'],
    'polarity_macro_f1': test_metrics['polarity_macro_f1'],
    'emotion_micro_f1': test_metrics['emotions']['micro_f1'],
    'emotion_macro_f1': test_metrics['emotions']['macro_f1'],
    'hamming_loss': test_metrics['emotions']['hamming_loss'],
}, name='synthetic smoke')

test_rows            10.00
polarity_macro_f1     1.00
emotion_micro_f1      1.00
emotion_macro_f1      0.75
hamming_loss          0.00
Name: synthetic smoke, dtype: float64

In [5]:
sample_prediction = predict(
    artifact_dir / 'model.joblib', '合格して本当にうれしい'
)
sample_prediction

{'polarity': 'positive',
 'polarity_confidence': 0.6603725066468707,
 'emotions': ['joy', 'trust'],
 'emotion_probabilities': {'emotion_joy': 0.7097243179440444,
  'emotion_sadness': 0.2532015508427731,
  'emotion_anticipation': 0.2588115051786891,
  'emotion_surprise': 0.2797244902113264,
  'emotion_anger': 0.18961671126505422,
  'emotion_fear': 0.25669375147442847,
  'emotion_disgust': 0.18961671126505422,
  'emotion_trust': 0.7475806194256442}}

## Takeaways

- Полярность и агрегированные multilabel-метрики на synthetic smoke согласуются с ожидаемыми шаблонами.
- Emotion macro-F1 = 0.750 при Hamming loss = 0.000: для классов без положительных test-примеров macro-F1 даёт ноль, поэтому обе метрики нужны вместе.
- Для содержательного эксперимента WRIME нужно получать напрямую у авторов и использовать только в рамках актуальных лицензионных ограничений.